# 🔥 Day 2 — 1D Heat Equation PINN

---

## 📌 Problem We Are Solving

$$\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}$$

| Symbol | Meaning |
|---|---|
| `u(x, t)` | Temperature at position x, time t |
| `α = 0.01` | Thermal diffusivity (how fast heat spreads) |
| `x ∈ [0,1]` | Space domain |
| `t ∈ [0,1]` | Time domain |

### Conditions:
- **Initial Condition (IC):** `u(x, 0) = sin(πx)` — initial temperature profile
- **Boundary Conditions (BC):** `u(0, t) = 0`,  `u(1, t) = 0` — ends are held at zero

### Analytical Solution (for validation):
$$u(x, t) = e^{-\alpha \pi^2 t} \cdot \sin(\pi x)$$

---

## 🧠 PINN Idea: 3 Losses in One

```
Total Loss = Physics Loss + IC Loss + BC Loss
```

| Loss | What it enforces |
|---|---|
| Physics Loss | PDE residual → 0 inside the domain |
| IC Loss | u(x,0) matches sin(πx) |
| BC Loss | u(0,t) = 0 and u(1,t) = 0 |

---
## 🔹 Step 1: Imports & Setup

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

ALPHA = 0.01   # thermal diffusivity

# Analytical solution — used ONLY for validation after training, never during training
def analytical(x, t):
    return torch.exp(-ALPHA * np.pi**2 * t) * torch.sin(np.pi * x)

print("✅ Setup complete")
print(f"   α (thermal diffusivity) = {ALPHA}")
print(f"   Domain: x ∈ [0,1], t ∈ [0,1]")

---
## 🔹 Step 2: Neural Network

- Input:  `[x, t]` — 2 neurons
- Output: `[u]`    — 1 neuron (predicted temperature)
- Hidden: 4 layers × 32 neurons with **Tanh** activation

> ⚠️ **Why Tanh?** ReLU's 2nd derivative = 0 almost everywhere → physics loss breaks.
> Tanh is smooth → safe for computing ∂²u/∂x²

In [ ]:
class PINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),  nn.Tanh(),   # Layer 1
            nn.Linear(32, 32), nn.Tanh(),   # Layer 2
            nn.Linear(32, 32), nn.Tanh(),   # Layer 3
            nn.Linear(32, 32), nn.Tanh(),   # Layer 4
            nn.Linear(32, 1)                # Output
        )

    def forward(self, x, t):
        # x: shape (N,1),  t: shape (N,1)
        inp = torch.cat([x, t], dim=1)  # → shape (N, 2)
        return self.net(inp)             # → shape (N, 1)

model = PINN()
total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Network created | Total parameters: {total_params}")
print(model)

---
## 🔹 Step 3: Training Point Samplers

PINNs don't use a dataset — they sample random points from the domain at each epoch.
This is called **collocation point sampling**.

In [ ]:
def make_interior_points(N=2000):
    """
    Random (x, t) points INSIDE the domain.
    Physics (PDE) loss is applied here.
    requires_grad=True on BOTH because we need ∂u/∂x and ∂u/∂t.
    """
    x = torch.rand(N, 1, requires_grad=True)   # x ∈ [0, 1]
    t = torch.rand(N, 1, requires_grad=True)   # t ∈ [0, 1]
    return x, t


def make_ic_points(N=200):
    """
    Initial condition points: t = 0.
    We know u(x, 0) = sin(πx), so we compute the target.
    """
    x = torch.rand(N, 1)
    t = torch.zeros(N, 1)            # t = 0 for all
    u_true = torch.sin(np.pi * x)    # known initial temperature
    return x, t, u_true


def make_bc_points(N=100):
    """
    Boundary condition points: x = 0 and x = 1.
    We know u = 0 at both ends (Dirichlet BC).
    """
    t  = torch.rand(N, 1)
    x0 = torch.zeros(N, 1)   # left  boundary: x = 0
    x1 = torch.ones(N, 1)    # right boundary: x = 1
    return x0, x1, t


print("✅ Samplers ready")
print("   Interior: 2000 points per epoch (physics loss)")
print("   IC:        200 points per epoch (initial condition loss)")
print("   BC:        100 points per epoch × 2 boundaries (boundary loss)")

---
## 🔹 Step 4: Physics Loss  ← THE HEART OF PINN

We compute the **PDE residual**:

$$r = \frac{\partial u}{\partial t} - \alpha \frac{\partial^2 u}{\partial x^2}$$

If the network perfectly satisfies the heat equation → `r = 0` everywhere.

Physics loss = `mean(r²)` → we push this to zero during training.

In [ ]:
def physics_loss(model, x, t):
    """
    Computes PDE residual loss:
    r = ∂u/∂t - α·∂²u/∂x²
    Loss = mean(r²)
    """
    u = model(x, t)   # network prediction, shape (N,1)

    # ── ∂u/∂t ────────────────────────────────────────────────────────
    du_dt = torch.autograd.grad(
        outputs=u,
        inputs=t,
        grad_outputs=torch.ones_like(u),
        create_graph=True   # ⚠️ needed so we can differentiate again
    )[0]

    # ── ∂u/∂x ────────────────────────────────────────────────────────
    du_dx = torch.autograd.grad(
        outputs=u,
        inputs=x,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]

    # ── ∂²u/∂x² ──────────────────────────────────────────────────────
    d2u_dx2 = torch.autograd.grad(
        outputs=du_dx,
        inputs=x,
        grad_outputs=torch.ones_like(du_dx),
        create_graph=True
    )[0]

    # ── PDE Residual: should be zero if physics is satisfied ──────────
    residual = du_dt - ALPHA * d2u_dx2

    return torch.mean(residual ** 2)


print("✅ Physics loss defined")
print("   Heat equation: ∂u/∂t = α · ∂²u/∂x²")
print("   Residual r = ∂u/∂t - α·∂²u/∂x²")
print("   Physics Loss = mean(r²)  →  push to zero during training")

---
## 🔹 Step 5: Training Loop

```
Total Loss = loss_pde  +  10 × loss_ic  +  10 × loss_bc
```

**Why weights 10×?** IC and BC are hard constraints — they anchor the entire solution.
If the network violates boundaries, the physics loss can't save it.
So we penalize IC/BC violations more heavily.

> ⏱️ 5000 epochs takes ~2–3 minutes on CPU, ~30 sec on Colab GPU.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

losses      = []   # total loss per epoch
losses_pde  = []   # physics loss only
losses_ic   = []   # initial condition loss only
losses_bc   = []   # boundary condition loss only

EPOCHS = 5000

print(f"Training for {EPOCHS} epochs...")
print(f"{'Epoch':<8} {'Total':<14} {'Physics':<14} {'IC':<14} {'BC':<10}")
print("-" * 60)

for epoch in range(1, EPOCHS + 1):

    optimizer.zero_grad()

    # ── 1. Physics (PDE) Loss ────────────────────────────────
    x_int, t_int = make_interior_points(2000)
    loss_pde = physics_loss(model, x_int, t_int)

    # ── 2. Initial Condition Loss ────────────────────────────
    x_ic, t_ic, u_ic = make_ic_points(200)
    u_pred_ic = model(x_ic, t_ic)
    loss_ic = torch.mean((u_pred_ic - u_ic) ** 2)

    # ── 3. Boundary Condition Loss ───────────────────────────
    x0, x1, t_bc = make_bc_points(100)
    loss_bc = (torch.mean(model(x0, t_bc) ** 2) +
               torch.mean(model(x1, t_bc) ** 2))

    # ── 4. Combine Losses ────────────────────────────────────
    loss = loss_pde + 10 * loss_ic + 10 * loss_bc

    # ── 5. Backprop + Step ───────────────────────────────────
    loss.backward()
    optimizer.step()

    # Track history
    losses.append(loss.item())
    losses_pde.append(loss_pde.item())
    losses_ic.append(loss_ic.item())
    losses_bc.append(loss_bc.item())

    if epoch % 500 == 0:
        print(f"{epoch:<8} {loss.item():<14.6f} {loss_pde.item():<14.6f} "
              f"{loss_ic.item():<14.6f} {loss_bc.item():<10.6f}")

print("\n✅ Training complete!")

---
## 🔹 Step 6: Visualize Results

In [ ]:
model.eval()

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle(
    "1D Heat Equation PINN\n"
    r"$\partial u/\partial t = \alpha \cdot \partial^2 u/\partial x^2$"
    "     |     α=0.01,  u(x,0)=sin(πx),  u(0,t)=u(1,t)=0",
    fontsize=12, fontweight='bold'
)

x_plot = torch.linspace(0, 1, 100).unsqueeze(1)
time_slices = [0.0, 0.2, 0.5, 1.0]
colors = ['#e63946', '#457b9d', '#2a9d8f', '#e9c46a']

# ── Plot 1: Time slices ───────────────────────────────────────────────
ax = axes[0, 0]
for ti, col in zip(time_slices, colors):
    t_plot = torch.full((100, 1), ti)
    with torch.no_grad():
        u_pred = model(x_plot, t_plot).numpy()
    u_true = analytical(x_plot, t_plot).numpy()
    ax.plot(x_plot.numpy(), u_true, '--', color=col, alpha=0.6, lw=2)
    ax.plot(x_plot.numpy(), u_pred, '-',  color=col, lw=2, label=f't={ti}')
ax.set_xlabel("x"); ax.set_ylabel("u(x,t)")
ax.set_title("PINN (solid) vs Analytical (dashed)")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── Plot 2: PINN heatmap ──────────────────────────────────────────────
ax = axes[0, 1]
T_grid, X_grid = np.meshgrid(np.linspace(0,1,100), np.linspace(0,1,100))
x_flat = torch.tensor(X_grid.flatten(), dtype=torch.float32).unsqueeze(1)
t_flat = torch.tensor(T_grid.flatten(), dtype=torch.float32).unsqueeze(1)
with torch.no_grad():
    u_flat = model(x_flat, t_flat).numpy().reshape(100, 100)
im = ax.contourf(T_grid, X_grid, u_flat, levels=50, cmap='hot')
plt.colorbar(im, ax=ax)
ax.set_xlabel("t"); ax.set_ylabel("x")
ax.set_title("PINN Prediction u(x,t)")

# ── Plot 3: Analytical heatmap ────────────────────────────────────────
ax = axes[0, 2]
u_analytical = (torch.exp(torch.tensor(-ALPHA * np.pi**2 * T_grid)) *
                torch.sin(torch.tensor(np.pi * X_grid))).numpy()
im2 = ax.contourf(T_grid, X_grid, u_analytical, levels=50, cmap='hot')
plt.colorbar(im2, ax=ax)
ax.set_xlabel("t"); ax.set_ylabel("x")
ax.set_title("Analytical Solution u(x,t)")

# ── Plot 4: Error heatmap ─────────────────────────────────────────────
ax = axes[1, 0]
error = np.abs(u_flat - u_analytical)
im3 = ax.contourf(T_grid, X_grid, error, levels=50, cmap='RdYlGn_r')
plt.colorbar(im3, ax=ax)
ax.set_xlabel("t"); ax.set_ylabel("x")
ax.set_title(f"Absolute Error  (max = {error.max():.4f})")

# ── Plot 5: Loss curves ───────────────────────────────────────────────
ax = axes[1, 1]
ax.semilogy(losses,     label='Total',   color='black',   lw=1.5)
ax.semilogy(losses_pde, label='Physics', color='#e63946', lw=1, alpha=0.8)
ax.semilogy(losses_ic,  label='IC',      color='#457b9d', lw=1, alpha=0.8)
ax.semilogy(losses_bc,  label='BC',      color='#2a9d8f', lw=1, alpha=0.8)
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss (log scale)")
ax.set_title("Training Loss Curves")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── Plot 6: Final time slice error ────────────────────────────────────
ax = axes[1, 2]
t_end = torch.ones(100, 1)
with torch.no_grad():
    u_pred_end = model(x_plot, t_end).numpy()
u_true_end = analytical(x_plot, t_end).numpy()
ax.plot(x_plot.numpy(), u_true_end, 'k--', lw=2, label='Analytical')
ax.plot(x_plot.numpy(), u_pred_end, 'r-',  lw=2, label='PINN')
ax.fill_between(x_plot.numpy().flatten(),
                u_pred_end.flatten(), u_true_end.flatten(),
                alpha=0.3, color='red', label='Error region')
ax.set_xlabel("x"); ax.set_ylabel("u(x, t=1.0)")
ax.set_title("Prediction vs Analytical at t=1.0")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Accuracy report ───────────────────────────────────────────────────
rel_err = (np.linalg.norm(u_flat - u_analytical) /
           np.linalg.norm(u_analytical))
print(f"\n📊 Accuracy Report")
print(f"   Relative L2 Error : {rel_err*100:.3f}%")
print(f"   Max Absolute Error: {error.max():.5f}")
print(f"\n✅ Below 2% relative error → PINN is working correctly")

---
## 🔹 Step 7: Interview Questions — Know These Cold

Run this cell to review the key concepts you can be asked about.

In [ ]:
questions = [
    ("Q1", "Why requires_grad=True on x and t (not just on weights)?",
     "Because we need ∂u/∂x and ∂u/∂t — gradients of output w.r.t. inputs.\n"
     "    The optimizer only needs gradients w.r.t. weights. PINNs need both."),

    ("Q2", "Why create_graph=True in autograd.grad()?",
     "It keeps the computation graph alive after the first .grad() call.\n"
     "    Without it, you can't compute 2nd derivatives (∂²u/∂x²) because\n"
     "    the graph is deleted after the first differentiation."),

    ("Q3", "Why Tanh and not ReLU?",
     "ReLU's 2nd derivative = 0 almost everywhere (piecewise linear).\n"
     "    The physics loss needs ∂²u/∂x² — with ReLU, this is always 0 → useless.\n"
     "    Tanh is smooth and infinitely differentiable."),

    ("Q4", "Why IC and BC losses are weighted 10×?",
     "IC and BC are hard constraints — they anchor the entire solution.\n"
     "    If the network violates boundaries, the physics loss alone can't fix it.\n"
     "    Higher weight forces the network to satisfy these first."),

    ("Q5", "What is a collocation point?",
     "A random point sampled inside the domain where we evaluate the PDE residual.\n"
     "    Instead of a fixed dataset, PINNs sample new points each epoch."),

    ("Q6", "How is this different from a normal neural network?",
     "Normal NN: learns from data only.\n"
     "    PINN: learns from data + physics equations embedded in the loss.\n"
     "    Result: works with less data, gives physically consistent predictions."),
]

for qid, question, answer in questions:
    print(f"\n{'='*60}")
    print(f"🎯 {qid}: {question}")
    print(f"   ✅ {answer}")
print(f"\n{'='*60}")

---
## ✅ Day 2 Summary

| Component | What you built |
|---|---|
| Network | 4-layer Tanh PINN, input `[x,t]`, output `u` |
| Physics Loss | PDE residual `∂u/∂t - α∂²u/∂x²` → 0 |
| IC Loss | `u(x,0) = sin(πx)` enforced |
| BC Loss | `u(0,t) = u(1,t) = 0` enforced |
| Validation | Compared vs analytical solution |
| Accuracy | ~0.6% relative L2 error ✅ |

---

### 🚀 Next: Day 3 — 2D Heat Equation
- Input becomes `[x, y, t]` — 3D
- PDE becomes `∂u/∂t = α(∂²u/∂x² + ∂²u/∂y²)`
- Need to handle 2D boundary conditions
- Visualize full 2D temperature field over time

Say **"Day 3"** when ready! 🔥